# Fair Lending Disparity Analysis — Worked Example

**Package:** fair-lending-screener v0.1.0  
**Methodology:** FFIEC Interagency Fair Lending Examination Procedures (2009)  
**Calibration target:** The Markup (2021) — adjusted OR 1.8× for Black vs. White, 2019 HMDA  

---

## What this notebook demonstrates

This notebook walks through a complete adjusted denial disparity analysis:

1. Load HMDA data from the CFPB public API (or from a local file)
2. Apply FFIEC-standard dataset filters
3. Run adjusted denial disparity analysis for multiple lenders
4. Generate journalist-legible Markdown reports
5. Interpret results correctly — including what they cannot prove

**Audience:** Community advocates, investigative journalists, fair housing researchers, CDFI compliance staff.

---

## ⚠ Before you start — read this

- Results from this tool are **screening signals**, not findings of discrimination
- Public HMDA data **does not include credit score, AUS recommendations, or asset data**
- A statistically significant adjusted disparity means the disparity warrants further review — not that the lender violated the law
- See `docs/limitations.md` for the complete list of what HMDA cannot tell you
- **Alpha release:** methodology peer review planned before v1.0.0

In [ ]:
# Install if needed
# !pip install fair-lending-screener

In [ ]:
import warnings
import fair_lending_screener as fls

print(f"fair-lending-screener version: {fls.__version__}")
print("Both import styles work:")
import fairlendingscreener
print(f"  import fairlendingscreener → version {fairlendingscreener.__version__}")

## Part 1: Using Synthetic Data (No Internet Required)

For demonstration and testing, `load_sample()` generates synthetic HMDA-like data
with all required fields. **Do not use this to draw conclusions about real lenders.**

Note: pseudo-R² on synthetic data is low (0.03–0.06) because MSA assignments are random
and carry no real geographic signal. On real HMDA, expect 0.15–0.25.

In [ ]:
# Load synthetic sample — all required fields included
raw_synthetic = fls.load_sample(n=3000, seed=42)

print(f"Records loaded: {len(raw_synthetic):,}")
print(f"Columns: {list(raw_synthetic.columns)}")
raw_synthetic.head(3)

In [ ]:
# Apply FFIEC-standard dataset filters:
# - Conventional first-lien home purchase (loan_type=1, lien_status=1, loan_purpose=1)
# - Site-built one-to-four unit principal residence
# - LTV ≤ 100%, action_taken ∈ {1, 3} (originated or denied)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df_synthetic = fls.prepare_for_analysis(raw_synthetic)

print(f"Records after FFIEC filters: {len(df_synthetic):,}")

# Raw denial rates by race
denial_by_race = (
    df_synthetic.groupby("derived_race")["is_denied"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "denial_rate", "count": "n"})
    .sort_values("denial_rate", ascending=False)
)
denial_by_race["denial_rate"] = (denial_by_race["denial_rate"] * 100).round(1)
print("\nRaw denial rates by race (uncontrolled):")
denial_by_race

In [ ]:
# Run the core analysis: adjusted denial disparity, Black vs. White
result_synthetic = fls.adjusted_denial_disparity(
    df_synthetic,
    protected_class="Black or African American",
    comparison_class="White",
)

print("=" * 60)
print("ADJUSTED DENIAL DISPARITY RESULT")
print("=" * 60)
print(f"Unadjusted odds ratio:  {result_synthetic.unadjusted_odds_ratio:.4f}×")
print(f"Adjusted odds ratio:    {result_synthetic.adjusted_odds_ratio:.4f}×")
print(f"95% CI:                 {result_synthetic.confidence_interval_95[0]:.4f}–{result_synthetic.confidence_interval_95[1]:.4f}×")
print(f"p-value:                {result_synthetic.p_value:.6f}")
print(f"Significant (p<0.05):   {result_synthetic.is_statistically_significant}")
print(f"Sample size:            {result_synthetic.sample_size:,} ({result_synthetic.sample_size_protected:,} Black, {result_synthetic.sample_size_comparison:,} White)")
print(f"Controls used:          {len(result_synthetic.controls_used)} (incl. {result_synthetic.model_diagnostics['n_msa_dummies']} MSA dummies)")
print(f"Pseudo-R²:              {result_synthetic.model_diagnostics['pseudo_r2_mcfadden']:.4f} (low — synthetic MSAs lack real geographic signal)")
if result_synthetic.dropped_controls:
    print(f"Dropped (zero-var):     {result_synthetic.dropped_controls}")

In [ ]:
# Interpretation — journalist-legible, no discrimination language
print(result_synthetic.interpretation)

In [ ]:
# Generate a full Markdown report
# Note: lender_name is suppressed in the headline when guardrails fire
# (e.g., p > 0.05, pseudo-R² < 0.05, non-convergence)
report = fls.generate_disparity_report(
    result_synthetic,
    geography="Synthetic Sample",
    year=2023,
    include_methodology=True,
)

# Display as rendered Markdown
try:
    from IPython.display import Markdown, display
    display(Markdown(report))
except ImportError:
    print(report)

---

## Part 2: Real HMDA Data from the CFPB API

This section requires internet access. It fetches real 2023 HMDA data for Illinois.

**Important:** Results from real data may name specific lenders. Per the safety guardrails
in `report.py`, lender names are suppressed in headlines when:
- p-value > 0.05 (not statistically significant)
- pseudo-R² < 0.05 (model underspecified)
- Model did not converge

Real data should produce pseudo-R² of 0.15–0.25 with the full control set.

In [ ]:
# Check API health before fetching
try:
    health = fls.check_data_source()
    print(f"CFPB API: reachable (HTTP {health['status_code']})")
    LIVE_DATA_AVAILABLE = True
except fls.DataSourceError as e:
    print(f"CFPB API unreachable: {e}")
    print("Using synthetic data for the remainder of this notebook.")
    LIVE_DATA_AVAILABLE = False

In [ ]:
if LIVE_DATA_AVAILABLE:
    # Fetch 2023 Illinois HMDA data (conventional first-lien home purchase)
    # This may take 30–60 seconds
    print("Fetching 2023 Illinois HMDA data...")
    raw_live = fls.load_from_api(year=2023, state="IL", limit=50_000)
    print(f"Loaded {len(raw_live):,} records")
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df_live = fls.prepare_for_analysis(raw_live)
    
    print(f"After FFIEC filters: {len(df_live):,} records")
    
    # Show denial rates by race
    denial_live = (
        df_live.groupby("derived_race")["is_denied"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "denial_rate", "count": "n"})
        .sort_values("denial_rate", ascending=False)
    )
    denial_live["denial_rate"] = (denial_live["denial_rate"] * 100).round(1)
    print("\nRaw denial rates by race (Illinois 2023, uncontrolled):")
    print(denial_live.to_string())
else:
    print("Skipped — no live data available.")

In [ ]:
if LIVE_DATA_AVAILABLE:
    # Run adjusted disparity analysis on all Illinois data (2023)
    try:
        result_live = fls.adjusted_denial_disparity(
            df_live,
            protected_class="Black or African American",
            comparison_class="White",
            min_sample_size=200,
        )
        
        print("ILLINOIS 2023 — ALL LENDERS COMBINED")
        print(f"Unadjusted OR: {result_live.unadjusted_odds_ratio:.3f}×")
        print(f"Adjusted OR:   {result_live.adjusted_odds_ratio:.3f}× (95% CI: {result_live.confidence_interval_95[0]:.3f}–{result_live.confidence_interval_95[1]:.3f}×)")
        print(f"p-value:       {result_live.p_value:.6f}")
        print(f"Significant:   {result_live.is_statistically_significant}")
        print(f"Pseudo-R²:     {result_live.model_diagnostics['pseudo_r2_mcfadden']:.4f}")
        print(f"Sample:        {result_live.sample_size:,} ({result_live.sample_size_protected:,} Black, {result_live.sample_size_comparison:,} White)")
        
        # Calibration check against The Markup (2021)
        adj_or = result_live.adjusted_odds_ratio
        in_range = 1.6 <= adj_or <= 2.2
        print(f"\nCalibration check (1.6–2.2× expected per docs/methodology.md): {'PASS' if in_range else 'INVESTIGATE'}")
        print("(The Markup found 1.8× nationally in 2019; our model may differ due to year, state, and control set)")
        
    except fls.InsufficientDataError as e:
        print(f"Insufficient data: {e}")
    except fls.ModelConvergenceError as e:
        print(f"Model convergence failure: {e}")
else:
    print("Skipped — no live data available.")

---

## Part 3: Error Handling — All Typed Exceptions

fair-lending-screener never returns a silent empty result. Every error path
raises a typed exception with a clear message.

In [ ]:
# InsufficientDataError — sample too small
tiny_df = df_synthetic.head(30)
try:
    fls.adjusted_denial_disparity(tiny_df, min_sample_size=100)
except fls.InsufficientDataError as e:
    print(f"InsufficientDataError: {e.actual} obs (need {e.minimum})")

In [ ]:
# InvalidProtectedClassError — class not in data
try:
    fls.adjusted_denial_disparity(df_synthetic, protected_class="Martian")
except fls.InvalidProtectedClassError as e:
    print(f"InvalidProtectedClassError: {e.requested!r} not found")
    print(f"Valid values: {e.valid_values}")

In [ ]:
# MissingControlsError — requested control column absent
try:
    fls.adjusted_denial_disparity(df_synthetic, controls=["nonexistent_credit_score_column"])
except fls.MissingControlsError as e:
    print(f"MissingControlsError: {e.missing_columns}")

---

## Part 4: Provenance — Reproducibility for Journalists and Attorneys

Every `DisparityResult` carries a `provenance` block with all information needed
to reproduce the analysis 18 months later.

In [ ]:
import json

print("Provenance block (copy verbatim when citing this result):")
print()
print(json.dumps(result_synthetic.provenance, indent=2, default=str))

---

## Part 5: What This Analysis Can and Cannot Prove

This is the most important section of the notebook.

### CAN show:
- A statistically significant disparity in denial rates after controlling for available legitimate factors
- The magnitude of that disparity (odds ratio) and its precision (confidence interval)
- Whether the disparity persists after controlling for income, LTV, DTI, property value, and MSA

### CANNOT show:
- That the lender **intentionally** discriminated (intent requires internal loan files)
- That the disparity is **caused** by race (causation requires experimental design)
- That the disparity would survive controlling for credit score or AUS (not in public HMDA)
- That any specific applicant was treated unlawfully
- That the lender violated ECOA or the Fair Housing Act

### Standard required language (appears in every generated report):

> This analysis identifies a statistically significant adjusted disparity in denial rates. It does not constitute a finding of discrimination under ECOA or the Fair Housing Act, and it does not establish that any protected class characteristic caused any lending outcome. Further review of application files, underwriting guidelines, and internal lender data would be required to assess whether discrimination occurred.

---

## Methodology Feedback

Open a GitHub issue tagged [`methodology`](https://github.com/Jaypatel1511/fair-lending-screener/issues)
with specific concerns and citations. All methodology changes are versioned in `CHANGELOG.md`.